# 🌐 Day 02a: OAuth2 — Authorization Flows & Social Login

---

## 🎯 What You'll Master Today
- OAuth2 concepts: roles, grant types, scopes
- Authorization Code flow (with PKCE)
- Client Credentials flow
- FastAPI OAuth2 scopes implementation
- Social login pattern (Google, GitHub)

---

## 🎭 OAuth2 — The Valet Key Analogy

Your car key opens everything — trunk, engine, doors. A **valet key** only opens the door and starts the engine (no trunk). OAuth2 is like giving apps a **valet key** to your data — limited access, revocable, without sharing your actual password.

```
╔═══════════════════════════════════════════════════════════════════╗
║  OAUTH2 ROLES                                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Resource Owner:    You (the user)                                ║
║  Client:            The app requesting access (3rd party app)    ║
║  Auth Server:       Issues tokens (Google, GitHub)               ║
║  Resource Server:   Has the data (Google API, GitHub API)        ║
║                                                                   ║
║  Grant Types (Flows):                                             ║
║  ─────────────────────                                            ║
║  1. Authorization Code  → Web apps (most secure, most common)   ║
║  2. Auth Code + PKCE    → Mobile/SPA (no client secret)         ║
║  3. Client Credentials  → Server-to-server (no user involved)   ║
║  4. Password Grant      → Legacy (deprecated, avoid)            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Authorization Code Flow — Step by Step
# ============================================================

print("""
Authorization Code Flow (most common for web apps):

┌──────────┐     ┌──────────────┐     ┌────────────────┐
│  Browser │     │  Your Server │     │  Google/GitHub  │
│ (Client) │     │  (Backend)   │     │  (Auth Server)  │
└────┬─────┘     └──────┬───────┘     └───────┬────────┘
     │                   │                      │
     │ 1. Click "Login   │                      │
     │    with Google"   │                      │
     │──────────────────>│                      │
     │                   │                      │
     │ 2. Redirect to    │                      │
     │    Google login   │                      │
     │<──────────────────│                      │
     │                   │                      │
     │ 3. User logs in to Google ───────────>   │
     │    "Allow access to email?"               │
     │                                           │
     │ 4. Google redirects back with AUTH CODE    │
     │<──────────────────────────────────────────│
     │                   │                      │
     │ 5. Send auth code │                      │
     │──────────────────>│                      │
     │                   │ 6. Exchange code      │
     │                   │    for tokens ───────>│
     │                   │                      │
     │                   │ 7. Get access_token   │
     │                   │<─────────────────────│
     │                   │                      │
     │                   │ 8. Use token to get  │
     │                   │    user info ————————>│
     │                   │                      │
     │ 9. Return user info + your JWT           │
     │<──────────────────│                      │
""")

In [ ]:
# ============================================================
# FastAPI OAuth2 Scopes — Role-Based Access Control
# ============================================================
from fastapi import FastAPI, Depends, HTTPException, Security, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm, SecurityScopes
from fastapi.testclient import TestClient
from jose import jwt
from pydantic import BaseModel
from datetime import datetime, timedelta
from passlib.context import CryptContext

SECRET_KEY = "secret-key-for-demo"
ALGORITHM = "HS256"

# Define scopes
oauth2_scheme = OAuth2PasswordBearer(
    tokenUrl="/auth/login",
    scopes={
        "read": "Read access to resources",
        "write": "Write access to resources",
        "admin": "Admin access — manage users",
    }
)

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

# Fake users with different scopes
users_db = {
    "sarthak": {
        "username": "sarthak",
        "password_hash": pwd_context.hash("admin123"),
        "scopes": ["read", "write", "admin"]
    },
    "viewer": {
        "username": "viewer",
        "password_hash": pwd_context.hash("view123"),
        "scopes": ["read"]
    }
}

def create_token(data: dict, scopes: list[str]):
    to_encode = data.copy()
    to_encode.update({
        "exp": datetime.utcnow() + timedelta(minutes=30),
        "scopes": scopes
    })
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

app = FastAPI()

async def get_current_user(security_scopes: SecurityScopes, token: str = Depends(oauth2_scheme)):
    """Verify token AND check required scopes."""
    credentials_exception = HTTPException(
        status_code=401,
        detail="Not authenticated",
        headers={"WWW-Authenticate": f"Bearer scope=\"{security_scopes.scope_str}\""},
    )
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username = payload.get("sub")
        token_scopes = payload.get("scopes", [])
    except Exception:
        raise credentials_exception
    
    # Check if token has all required scopes
    for scope in security_scopes.scopes:
        if scope not in token_scopes:
            raise HTTPException(
                status_code=403,
                detail=f"Missing required scope: {scope}"
            )
    
    return users_db.get(username)

@app.post("/auth/login")
def login(form: OAuth2PasswordRequestForm = Depends()):
    user = users_db.get(form.username)
    if not user or not pwd_context.verify(form.password, user["password_hash"]):
        raise HTTPException(401, "Invalid credentials")
    token = create_token({"sub": user["username"]}, user["scopes"])
    return {"access_token": token, "token_type": "bearer"}

# Scoped endpoints
@app.get("/items")
def read_items(user=Security(get_current_user, scopes=["read"])):
    return {"items": ["item1", "item2"], "user": user["username"]}

@app.post("/items")
def create_item(user=Security(get_current_user, scopes=["write"])):
    return {"message": "Item created", "user": user["username"]}

@app.delete("/users/{user_id}")
def delete_user(user_id: int, user=Security(get_current_user, scopes=["admin"])):
    return {"message": f"User {user_id} deleted by {user['username']}"}

# Test
client = TestClient(app)

# Admin login
r = client.post("/auth/login", data={"username": "sarthak", "password": "admin123"})
admin_token = r.json()["access_token"]

# Viewer login
r = client.post("/auth/login", data={"username": "viewer", "password": "view123"})
viewer_token = r.json()["access_token"]

# Admin can do everything
r = client.get("/items", headers={"Authorization": f"Bearer {admin_token}"})
print(f"Admin read: {r.status_code} ✅")

r = client.delete("/users/1", headers={"Authorization": f"Bearer {admin_token}"})
print(f"Admin delete: {r.status_code} ✅")

# Viewer can only read
r = client.get("/items", headers={"Authorization": f"Bearer {viewer_token}"})
print(f"Viewer read: {r.status_code} ✅")

r = client.post("/items", headers={"Authorization": f"Bearer {viewer_token}"})
print(f"Viewer write: {r.status_code} ❌ — {r.json()['detail']}")

In [ ]:
# ============================================================
# Social Login Pattern — Google OAuth2
# ============================================================

print("""
# Google OAuth2 in FastAPI (using authlib)
# pip install authlib httpx

from authlib.integrations.starlette_client import OAuth
from starlette.config import Config

config = Config('.env')  # Load from .env file
oauth = OAuth(config)

# Register Google
oauth.register(
    name='google',
    server_metadata_url='https://accounts.google.com/.well-known/openid-configuration',
    client_kwargs={'scope': 'openid email profile'},
)

# .env file:
# GOOGLE_CLIENT_ID=your-client-id.apps.googleusercontent.com
# GOOGLE_CLIENT_SECRET=your-secret

@app.get("/auth/google")
async def google_login(request: Request):
    redirect_uri = request.url_for('google_callback')
    return await oauth.google.authorize_redirect(request, redirect_uri)

@app.get("/auth/google/callback")
async def google_callback(request: Request):
    token = await oauth.google.authorize_access_token(request)
    user_info = token.get('userinfo')
    
    # user_info contains: email, name, picture, sub (Google user ID)
    # → Find or create user in your DB
    # → Issue your own JWT
    
    our_jwt = create_access_token({"sub": user_info["email"]})
    return {"access_token": our_jwt, "user": user_info}
""")

print("Social Login Flow:")
print("  1. User clicks 'Login with Google'")
print("  2. Redirect to Google's consent page")
print("  3. User authorizes, Google redirects back with code")
print("  4. Backend exchanges code for Google access token")
print("  5. Backend uses Google token to get user info")
print("  6. Backend creates/finds user, issues YOUR JWT")
print("  7. Frontend stores YOUR JWT, uses it for all API calls")

---

## 🗺️ OAuth2 Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  OAUTH2 PATTERNS                                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  AUTH CODE:    User login → code → exchange for tokens           ║
║  PKCE:         Auth Code + code_verifier (mobile/SPA)            ║
║  CLIENT CRED:  Server-to-server, no user involved                ║
║                                                                   ║
║  SCOPES:       OAuth2PasswordBearer(scopes={...})                ║
║  PROTECT:      Security(get_user, scopes=["read"])               ║
║  CHECK:        for scope in required_scopes: verify              ║
║                                                                   ║
║  SOCIAL:       authlib → register provider → redirect → callback║
║  FLOW:         Google token → user info → your JWT               ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Explain OAuth2 flow | Auth code: redirect → consent → code → exchange → tokens → API calls |
| 2 | OAuth2 vs JWT? | OAuth2 = protocol (how to get tokens). JWT = token format (how tokens look) |
| 3 | What is PKCE? | Proof Key for Code Exchange. Prevents auth code interception (mobile/SPA) |
| 4 | What are scopes? | Permissions like "read", "write", "admin". Limit what a token can access |
| 5 | Client credentials flow? | Server-to-server. No user. App authenticates with client_id + client_secret |
| 6 | How social login works? | Get code from provider → exchange for provider token → get user info → issue your JWT |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • OAuth2 = authorization protocol (how to get tokens)  │
## │  • Auth Code flow is most common (web apps)             │
## │  • PKCE for mobile/SPA (no client secret exposed)       │
## │  • Scopes control granular access (read/write/admin)    │
## │  • Security() enables scope-based protection            │
## │  • Social login: provider token → user info → your JWT  │
## │  • Use authlib for social login in FastAPI               │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Security Best Practices** — XSS, CSRF, SQL injection, and OWASP Top 10